In [6]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder


In [7]:
torch.tensor([1,2,3],dtype=float)


tensor([1., 2., 3.], dtype=torch.float64)

In [27]:
x=torch.tensor([3],dtype=torch.float32,requires_grad=True)
y=x**2


In [28]:
y.backward()

In [31]:
x.grad

tensor([6.])

In [88]:
import torch.nn as nn
class Model(nn.Module):
    def __init__(self,num_features):
        super().__init__()
        self.network=nn.Sequential(
            nn.Linear(num_features,3),
            nn.ReLU(),
            nn.Linear(3,1),
            nn.Sigmoid()
        )
        

    def forward(self,features):
        out=self.network(features)
        return out
    

    
    
        

    

In [89]:

features=torch.rand(10,5)
model=Model(5)



In [90]:
y_true = torch.randint(0,2,(10,1)).float()
y_pred=model(features)
y_pred.shape

torch.Size([10, 1])

In [91]:

eps = 1e-7

loss = -(y_true * torch.log(y_pred + eps) +
         (1 - y_true) * torch.log(1 - y_pred + eps))
loss

tensor([[0.5867],
        [0.6026],
        [0.7945],
        [0.8199],
        [0.7735],
        [0.5821],
        [0.6258],
        [0.5827],
        [0.6064],
        [0.6200]], grad_fn=<NegBackward0>)

In [92]:
model.linear1.bias

AttributeError: 'Model' object has no attribute 'linear1'

In [93]:
!pip install torchinfo

In [94]:
from torchinfo import summary
summary(model,input_size=(10,5))

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [10, 1]                   --
├─Sequential: 1-1                        [10, 1]                   --
│    └─Linear: 2-1                       [10, 3]                   18
│    └─ReLU: 2-2                         [10, 3]                   --
│    └─Linear: 2-3                       [10, 1]                   4
│    └─Sigmoid: 2-4                      [10, 1]                   --
Total params: 22
Trainable params: 22
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

In [1]:

from sklearn.datasets import make_classification
import torch

In [20]:
x_train,y_train=make_classification(
    n_samples=10,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_classes=2,
    random_state=42,
)

x_test,y_test=make_classification(
    n_samples=10,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_classes=2,
    random_state=42,
)





In [21]:
x_train=torch.tensor(x_train,dtype=torch.float32)
y_train=torch.tensor(y_train,dtype=torch.float32)


x_test=torch.tensor(x_test,dtype=torch.float32)
y_test=torch.tensor(y_test,dtype=torch.float32)



In [22]:
from torch.utils.data import DataLoader,Dataset

class CustomDataSet(Dataset):
    def __init__(self,features,labels):
        self.features=features
        self.labels=labels
    
    def __len__(self):
        return self.features.shape[0]

    def __getitem__(self, index):
        return self.features[index],self.labels[index]



In [23]:
train_dataset=CustomDataSet(x_train,y_train)
test_dataset=CustomDataSet(x_test,y_test)



In [25]:
train_dataloader=DataLoader(train_dataset,batch_size=2,shuffle=True)
test_dataloader=DataLoader(test_dataset,batch_size=2,shuffle=True)

In [27]:
import torch.nn as nn
class SimpleNN(nn.Module):
    def __init__(self,num_features):
        super().__init__()
        self.linear=nn.Linear(num_features,1)
        self.sigmoid=nn.Sigmoid()
    
    def forward(self,features):
        out=self.linear(features)
        out=self.sigmoid(out)
        return out


learning_rate=0.1
epochs=10

model=SimpleNN(x_train.shape[1])
optimizer=torch.optim.SGD(model.parameters(),lr=learning_rate)
loss_function=nn.BCELoss()




In [30]:
for epoch in range(epochs):
    for batch_feat,batch_lab in train_dataloader:
        y_pred=model(batch_feat)
        loss=loss_function(y_pred,batch_lab.view(-1,1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(loss.item())



0.04799571633338928
0.11155184358358383
0.0964568629860878
0.03770619258284569
0.1556546539068222
0.15935318171977997
0.13417308032512665
0.03714539855718613
0.023452915251255035
0.18628981709480286


In [31]:
model.eval()
accuracy_list=[]
with torch.no_grad():
    for batch_feat,batch_lab in test_dataloader:
        y_pred=model(batch_feat)
        y_pred=(y_pred>0.5).float()
        batch_acc=(y_pred.view(-1)==batch_lab).float().mean().item()
        accuracy_list.append(batch_acc)

print(accuracy_list)
print(sum(accuracy_list)/len(accuracy_list))



[1.0, 1.0, 1.0, 1.0, 0.5]
0.9
